# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/capstone/starter_B_pathmnist.ipynb)

# BME i9400 — Capstone starter B
## Tissue Classification in Colorectal Cancer Histology

This notebook downloads the images, shows the fixed split, and fits the baseline. Run it from top to
bottom before you change anything; it takes about two minutes.

---
## 1 — The data

PathMNIST, from the MedMNIST v2 collection (Yang et al., "MedMNIST v2", *Scientific Data*, 2023),
built from the colorectal cancer histology datasets of Kather et al. ("Predicting survival from
colorectal cancer histology slides using deep learning", *PLOS Medicine*, 2019). CC BY 4.0.

- **A row of $X$**: one image patch of hematoxylin and eosin (H&E) stained tissue, resized to
  28 × 28 pixels, three colour channels.
- **$y$**: one of nine tissue classes.

| Label | Tissue |
|---:|---|
| 0 | adipose |
| 1 | background |
| 2 | debris |
| 3 | lymphocytes |
| 4 | mucus |
| 5 | smooth muscle |
| 6 | normal colon mucosa |
| 7 | cancer-associated stroma |
| 8 | colorectal adenocarcinoma epithelium |

> **H&E staining** colours cell nuclei blue-purple (hematoxylin) and cytoplasm and connective tissue
> pink (eosin). Each patch is a small square cut from a whole-slide image of a tissue section.

In [ ]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

URL = "https://zenodo.org/records/10519652/files/pathmnist.npz?download=1"
z = np.load(io.BytesIO(urllib.request.urlopen(URL).read()))          # about 200 MB
X_train, y_train = z["train_images"], z["train_labels"].ravel()
X_val,   y_val   = z["val_images"],   z["val_labels"].ravel()
X_test,  y_test  = z["test_images"],  z["test_labels"].ravel()
TISSUES = ["adipose", "background", "debris", "lymphocytes", "mucus", "smooth muscle",
           "normal mucosa", "stroma", "tumour epithelium"]

counts = pd.DataFrame({s: np.bincount(y, minlength=9) for s, y in
                       [("training", y_train), ("validation", y_val), ("test", y_test)]}, index=TISSUES)
print(counts.to_string())

fig, axes = plt.subplots(3, 9, figsize=(13, 4.6))
for k in range(9):
    for row, i in enumerate(np.flatnonzero(y_train == k)[:3]):
        axes[row, k].imshow(X_train[i]); axes[row, k].axis("off")
    axes[0, k].set_title(TISSUES[k], fontsize=9)
plt.tight_layout(); plt.show()

---
## 2 — The fixed split

The official MedMNIST split is used unchanged. It has a property that matters for every result you
report:

- the **training** and **validation** sets (89,996 and 10,004 patches) were divided at random from
  one collection, NCT-CRC-HE-100K, cut from 86 slides; patches from the same slide appear in both
- the **test** set (7,180 patches) is a separate collection, CRC-VAL-HE-7K, from 50 patients who do
  not appear in the training data

> The validation set therefore measures performance on new patches from slides the model has already
> seen, and the test set measures performance on new patients. Expect validation scores to be higher
> than test scores. The class proportions also differ: the test set has more adipose and tumour
> epithelium, and less debris, than the training set.
>
> The **primary metric** is the balanced accuracy, the mean of the nine per-class sensitivities,
> which does not depend on the class proportions.

---
## 3 — The baseline

Multinomial logistic regression on the pixel values, after averaging each 2 × 2 block of pixels
(14 × 14 × 3 = 588 features) and standardising, with a fixed budget of 200 iterations of the
optimiser.

> The 2 × 2 averaging and the iteration budget keep the baseline's fitting time under a minute. The
> optimiser has not fully converged at 200 iterations; a longer run does not improve the validation
> score.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix
warnings.filterwarnings("ignore", category=ConvergenceWarning)

def pooled_pixels(images):
    # average each 2 x 2 block: (n, 28, 28, 3) -> (n, 14 * 14 * 3), scaled to [0, 1]
    n = len(images)
    return (images.reshape(n, 14, 2, 14, 2, 3).mean(axis=(2, 4)).reshape(n, -1) / 255).astype(np.float32)

baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=200))
baseline.fit(pooled_pixels(X_train), y_train)

pred_val_base = baseline.predict(pooled_pixels(X_val))
print(f"baseline, validation: balanced accuracy {balanced_accuracy_score(y_val, pred_val_base):.3f}")
print("sensitivity per class:", dict(zip(TISSUES, np.round(recall_score(y_val, pred_val_base, average=None), 3).tolist())))
print(pd.DataFrame(confusion_matrix(y_val, pred_val_base), index=TISSUES, columns=TISSUES).to_string())

---
## 4 — The test set and the bootstrap

> **The test set is used once.** The cell below defines the test-set evaluation and the bootstrap
> confidence interval but does not run them. Run them at the end of the project, for your final model
> and for the baseline only, after every other decision has been made on validation data.

> PathMNIST does not record which patient a test patch came from, so the bootstrap resamples
> patches. Patches from the same patient are not independent, so the interval is narrower than an
> interval that resampled patients would be; state this in the report.

In [ ]:
def bootstrap_difference(metric, y, pred_new, pred_base, groups=None, n_boot=2000, seed=0):
    """95% bootstrap confidence interval for metric(y, pred_new) - metric(y, pred_base).

    Resamples whole groups (patients, records, abstracts) with replacement when `groups` is given,
    otherwise single examples. Both models are scored on the same resample, so the interval is for
    the paired difference (Meeting 20).
    """
    rng = np.random.default_rng(seed)
    y, pred_new, pred_base = np.asarray(y), np.asarray(pred_new), np.asarray(pred_base)
    if groups is None:
        groups = np.arange(len(y))
    uniq, inv = np.unique(np.asarray(groups), return_inverse=True)
    members = [np.flatnonzero(inv == k) for k in range(len(uniq))]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        diffs.append(metric(y[idx], pred_new[idx]) - metric(y[idx], pred_base[idx]))
    point = metric(y, pred_new) - metric(y, pred_base)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return point, lo, hi


def evaluate_on_test(pred_test_new, pred_test_base):
    """Balanced accuracy of both models on the test set, and the 95% CI for the difference, resampling patches."""
    for name, p in [("final model", pred_test_new), ("baseline", pred_test_base)]:
        print(f"{name:12s} balanced accuracy {balanced_accuracy_score(y_test, p):.3f}")
    d, lo, hi = bootstrap_difference(balanced_accuracy_score, y_test, pred_test_new, pred_test_base)
    print(f"difference in balanced accuracy {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]")

# At the end of the project, and only then:
# evaluate_on_test(final_model_predictions_on_X_test, baseline.predict(pooled_pixels(X_test)))

### Computing time

A small convolutional network (three convolutional layers) trains at about one to two minutes per
pass over the 89,996 training patches on a free Colab CPU. Save trained weights to Google Drive with
`torch.save` so that a model is trained once, and subsample the training set while exploring.

---
## 5 — From here

The requirements, deliverables, and grading are in the capstone brief,
[`capstone/README.md`](https://github.com/dmochow/BME-i9400-2026/blob/main/capstone/README.md). In
short: reproduce the baseline above; search at least two further model families with every choice
made on validation data; evaluate the final model and the baseline once on the test set, with a 95%
bootstrap confidence interval for the difference; analyse the errors; write the model card.

Make a copy of this notebook (`File ▸ Save a copy in Drive`) and build your study in it.

Directions specific to this dataset: a small CNN (Meeting 21); data augmentation (flips and
90-degree rotations, which do not change the tissue class, and colour jitter, which imitates
differences in staining between laboratories); a pretrained image encoder, frozen or fine-tuned, on
images resized up to its input size (Meeting 22); Grad-CAM in the error analysis.